# Voice Spoof Detection — WavLM + Wav2Vec2 (Colab, HF dataset)

Akademik PoC. ASVspoof 2019 LA (`Bisher/ASVspoof_2019_LA` HF deposu) üzerinde iki SSL encoder eğitilir, kalibre edilir ve late fusion ile birleştirilir.

**Runtime:** GPU (T4 yeterli). Veri seti otomatik gelir, manuel indirme yok.

## 1. Repo + bağımlılıklar

In [ ]:
%cd /content
# Repo zaten clone edildiyse atla.
import os
if not os.path.isdir('voice-spoof-detection'):
    !git clone https://example.com/your/voice-spoof-detection.git
%cd voice-spoof-detection
!pip install -q -r requirements.txt

## 2. HF dataset parametreleri

`HF_STREAMING = True` → diske ~7.5 GB indirmez, parquet shard'larını akıtır (epoch boyu `streaming_steps_per_epoch` ile sınırlandırılır).

`HF_STREAMING = False` → ilk çağrıda HF cache'e ~7.5 GB indirir; sonraki çalıştırmalar offline.

In [ ]:
HF_REPO = 'Bisher/ASVspoof_2019_LA'
HF_STREAMING = False
CFG_WAVLM = 'configs/wavlm_hf.yaml'
CFG_W2V   = 'configs/wav2vec2_hf.yaml'

## 3. Configleri patch'le (HF repo + streaming)

In [ ]:
import yaml
for cfg_path in (CFG_WAVLM, CFG_W2V):
    cfg = yaml.safe_load(open(cfg_path))
    cfg['data']['hf_repo'] = HF_REPO
    cfg['data']['streaming'] = HF_STREAMING
    yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
    print('patched', cfg_path, '→ source=', cfg['data']['source'], 'repo=', cfg['data']['hf_repo'], 'streaming=', cfg['data']['streaming'])

## 4. Veriden bir örnek dinle (sanity check)

In [ ]:
from datasets import load_dataset, Audio
import IPython.display as ipd
ds = load_dataset(HF_REPO, split='train', streaming=True)
ds = ds.cast_column('audio', Audio(sampling_rate=16000))
row = next(iter(ds))
print('Columns:', list(row.keys()))
print('Label:', row.get('key'), 'System:', row.get('system_id'), 'Speaker:', row.get('speaker_id'))
aud = row['audio']
if isinstance(aud, dict):
    arr, sr = aud['array'], aud['sampling_rate']
else:
    s = aud.get_all_samples(); arr, sr = s.data, s.sample_rate
ipd.display(ipd.Audio(arr, rate=sr))

## 5. WavLM eğitimi

In [ ]:
!python -m src.train --config $CFG_WAVLM

## 6. Wav2Vec2 eğitimi

In [ ]:
!python -m src.train --config $CFG_W2V

## 7. Evaluation + Calibration + Fusion

In [ ]:
import yaml
WAVLM_CKPT = yaml.safe_load(open(CFG_WAVLM))['training']['save_dir'] + '/best.pt'
W2V_CKPT   = yaml.safe_load(open(CFG_W2V))['training']['save_dir']   + '/best.pt'
print('Loading:', WAVLM_CKPT, '|', W2V_CKPT)
!python -m src.evaluate \
    --wavlm-checkpoint   $WAVLM_CKPT \
    --wav2vec2-checkpoint $W2V_CKPT \
    --fusion

## 8. Grafikler + karşılaştırma tablosu

In [ ]:
import pandas as pd, IPython.display as ipd, os
out = 'outputs/evaluation'
if os.path.exists(f'{out}/comparison.csv'):
    ipd.display(pd.read_csv(f'{out}/comparison.csv'))
for path in [f'{out}/wavlm/confusion_matrix.png',
             f'{out}/wav2vec2/confusion_matrix.png',
             f'{out}/fusion/confusion_matrix.png',
             f'{out}/fusion/roc.png',
             f'{out}/wavlm/training_history.png',
             f'{out}/wav2vec2/training_history.png']:
    if os.path.exists(path):
        ipd.display(ipd.Image(path))

## 9. Gradio demo

In [ ]:
!python app.py \
    --wavlm-checkpoint   $WAVLM_CKPT \
    --wav2vec2-checkpoint $W2V_CKPT \
    --fusion-bundle      outputs/evaluation/fusion/fusion_bundle.json \
    --share